# Robust Hybrid Safety Pipeline

EMPTY nghĩa là không có người/vật, thuộc nhóm non-hazard nhưng vẫn giữ là lớp riêng. Chỉ dòng thiếu left_cm hoặc right_cm bị loại. APPROACHING, DANGER và RETREATING là hazard.


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesClassifier, IsolationForest
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score, precision_score, recall_score

RANDOM_STATE = 42
LABELS = ['SAFE', 'APPROACHING', 'DANGER', 'RETREATING', 'EMPTY']
HAZARD_LABELS = {'APPROACHING', 'DANGER', 'RETREATING'}
FEATURE_VERSION = 'robust-v2-drop-missing-sensor'

def find_data_dir():
    candidates = [Path.cwd() / 'ai_model' / 'data' / 'raw', Path.cwd().parent / 'data' / 'raw', Path('/kaggle/input')]
    for root in candidates:
        if root.is_dir() and list(root.glob('*.csv')):
            return root
        if root.is_dir():
            matches = list(root.rglob('feature_engineering.py'))
            if matches and (matches[0].parent / 'data' / 'raw').is_dir():
                return matches[0].parent / 'data' / 'raw'
    raise FileNotFoundError('Không tìm thấy thư mục CSV raw data')

DATA_DIR = find_data_dir()
print('DATA_DIR:', DATA_DIR)


In [ ]:
RAW_LABELS = set(LABELS)
FEATURE_COLUMNS = ['left_cm', 'right_cm', 'min_distance_cm', 'distance_delta_cm', 'sensor_spread_cm', 'distance_median_5', 'distance_std_5', 'approach_speed_cm_s', 'approach_rate_cm_s', 'bad_dt']

def add_robust_features(frame):
    required = {'timestamp', 'elapsed_s', 'left_cm', 'right_cm', 'label', 'source_file'}
    missing = required.difference(frame.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    out = frame.copy()
    for col in ['left_cm', 'right_cm', 'elapsed_s']:
        out[col] = pd.to_numeric(out[col], errors='coerce')
    out['label'] = out['label'].astype(str).str.upper().str.strip()
    unknown = sorted(set(out['label'].dropna()) - RAW_LABELS)
    if unknown:
        raise ValueError(f'Unknown labels: {unknown}')
    before = len(out)
    out = out.sort_values(['source_file', 'elapsed_s'], kind='stable')
    keep = out['elapsed_s'].notna() & out[['left_cm', 'right_cm']].notna().all(axis=1)
    out = out.loc[keep].copy().reset_index(drop=True)
    out.attrs['dropped_missing_sensor'] = int(before - len(out))
    out['min_distance_cm'] = out[['left_cm', 'right_cm']].min(axis=1)
    out['distance_delta_cm'] = out['left_cm'] - out['right_cm']
    out['sensor_spread_cm'] = out['distance_delta_cm'].abs()
    grouped = out.groupby('source_file', sort=False)
    out['dt_s'] = grouped['elapsed_s'].diff()
    valid_dt = out['dt_s'].between(0.03, 1.0, inclusive='both')
    out['bad_dt'] = (~valid_dt).astype(int)
    out['distance_median_5'] = grouped['min_distance_cm'].transform(lambda s: s.rolling(5, min_periods=1).median())
    out['distance_std_5'] = grouped['min_distance_cm'].transform(lambda s: s.rolling(5, min_periods=2).std()).fillna(0.0)
    smooth_delta = grouped['distance_median_5'].diff()
    out['approach_speed_cm_s'] = smooth_delta.div(out['dt_s'].where(valid_dt)).replace([np.inf, -np.inf], np.nan).clip(-500.0, 500.0).fillna(0.0)
    out['approach_rate_cm_s'] = -out['approach_speed_cm_s']
    return out

def load_recordings(data_dir):
    frames = []
    for path in sorted(Path(data_dir).glob('*.csv')):
        frame = pd.read_csv(path)
        frame['source_file'] = path.name
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(f'No CSV files in {data_dir}')
    return add_robust_features(pd.concat(frames, ignore_index=True))

data = load_recordings(DATA_DIR)
print('Rows after dropping incomplete sensors:', len(data))
print('Dropped rows:', data.attrs['dropped_missing_sensor'])
print('Files:', data['source_file'].nunique())
print(data['label'].value_counts().reindex(LABELS, fill_value=0))
print('Bad dt rows:', int(data['bad_dt'].sum()))


## Group split and training

EMPTY được giữ là lớp NO_OBJECT/non-hazard. Nếu chỉ có một recording EMPTY, notebook chỉ có thể train lớp này; cần thêm recording độc lập để đánh giá tổng quát hóa.


In [ ]:
def choose_group_split(frame, test_size, random_state, required_test=(), required_train=()):
    labels = frame['label'].reset_index(drop=True)
    groups = frame['source_file'].reset_index(drop=True)
    overall = labels.value_counts(normalize=True)
    splitter = GroupShuffleSplit(n_splits=500, test_size=test_size, random_state=random_state)
    best, best_score = None, float('inf')
    for train_idx, test_idx in splitter.split(frame, labels, groups=groups):
        train_labels, test_labels = set(labels.iloc[train_idx]), set(labels.iloc[test_idx])
        missing = len(set(required_train) - train_labels) + len(set(required_test) - test_labels)
        balance = sum(abs((labels.iloc[test_idx] == c).mean() - overall.get(c, 0.0)) for c in required_test)
        score = 100.0 * missing + abs(len(test_idx) / len(frame) - test_size) + balance
        if score < best_score:
            best_score, best = score, (train_idx, test_idx)
    if best is None:
        raise RuntimeError('Không tạo được group split')
    return best

required_test = {'SAFE', 'APPROACHING', 'DANGER', 'RETREATING'}
train_pool_idx, test_idx = choose_group_split(data, 0.20, RANDOM_STATE, required_test, set(LABELS))
train_pool = data.iloc[train_pool_idx].reset_index(drop=True)
test = data.iloc[test_idx].reset_index(drop=True)
train_idx, val_idx = choose_group_split(train_pool, 0.25, RANDOM_STATE + 1, required_test, set(LABELS))
train = train_pool.iloc[train_idx].reset_index(drop=True)
validation = train_pool.iloc[val_idx].reset_index(drop=True)
for name, part in [('train', train), ('validation', validation), ('test', test)]:
    print(f'{name}: rows={len(part)}, files={part.source_file.nunique()}')
    print(part['label'].value_counts().reindex(LABELS, fill_value=0))
print('Group overlaps:', set(train.source_file) & set(validation.source_file), set(train.source_file) & set(test.source_file), set(validation.source_file) & set(test.source_file))

X_train, y_train = train[FEATURE_COLUMNS], train['label']
X_val, y_val = validation[FEATURE_COLUMNS], validation['label']
X_test, y_test = test[FEATURE_COLUMNS], test['label']
classifier = ExtraTreesClassifier(n_estimators=600, min_samples_leaf=2, class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1)
classifier.fit(X_train, y_train)
normal_train = X_train[y_train == 'SAFE']
anomaly_models = {}
for contamination in [0.03, 0.05, 0.10]:
    model = Pipeline([('scale', StandardScaler()), ('detector', IsolationForest(n_estimators=400, contamination=contamination, random_state=RANDOM_STATE, max_features=1.0))])
    model.fit(normal_train)
    anomaly_models[contamination] = model
print('Classifier classes:', classifier.classes_)
print('SAFE-only anomaly rows:', len(normal_train))
print(classification_report(y_val, classifier.predict(X_val), labels=LABELS, target_names=LABELS, zero_division=0))


In [ ]:
def hybrid_predict(features, classifier, anomaly_model, warning_cm, danger_cm, confidence_threshold, logic='and'):
    pred = classifier.predict(features)
    confidence = classifier.predict_proba(features).max(axis=1)
    anomaly = anomaly_model.predict(features) == -1
    minimum = features['min_distance_cm'].to_numpy()
    states, reasons = [], []
    for i in range(len(features)):
        if pred[i] == 'EMPTY':
            states.append('EMPTY'); reasons.append('NO_OBJECT'); continue
        if minimum[i] <= danger_cm:
            states.append('EMERGENCY'); reasons.append('HARD_DANGER_THRESHOLD'); continue
        if minimum[i] <= warning_cm:
            states.append('WARNING'); reasons.append('HARD_WARNING_THRESHOLD'); continue
        classifier_vote = pred[i] in HAZARD_LABELS and confidence[i] >= confidence_threshold
        anomaly_vote = bool(anomaly[i])
        use_ai = (logic == 'and' and classifier_vote and anomaly_vote) or (logic == 'or' and (classifier_vote or anomaly_vote)) or (logic == 'classifier' and classifier_vote)
        states.append('WARNING' if use_ai else 'SAFE')
        reasons.append('AI_' + logic.upper() if use_ai else 'NORMAL')
    return pd.DataFrame({'state': states, 'reason': reasons, 'classifier_state': pred, 'classifier_confidence': confidence, 'anomaly_flag': anomaly.astype(int)}, index=features.index)

def safety_metrics(states, target):
    actual = target.isin(HAZARD_LABELS).to_numpy().astype(int)
    predicted = states.isin(['WARNING', 'EMERGENCY']).to_numpy().astype(int)
    cm = confusion_matrix(actual, predicted, labels=[0, 1])
    non_hazard = actual == 0
    return {'accuracy_hazard': float(accuracy_score(actual, predicted)), 'balanced_accuracy_hazard': float(balanced_accuracy_score(actual, predicted)), 'precision_hazard': float(precision_score(actual, predicted, zero_division=0)), 'recall_hazard': float(recall_score(actual, predicted, zero_division=0)), 'f1_hazard': float(f1_score(actual, predicted, zero_division=0)), 'non_hazard_fpr': float(predicted[non_hazard].mean()) if non_hazard.any() else 0.0, 'empty_recall': float(((states == 'EMPTY') & (target == 'EMPTY')).sum() / max((target == 'EMPTY').sum(), 1)), 'confusion_matrix': cm.tolist()}


In [ ]:
rows = []
for contamination, anomaly_model in anomaly_models.items():
    for logic in ['and', 'classifier', 'or']:
        for confidence in [0.40, 0.55, 0.70, 0.85]:
            for danger in [28.0, 30.0, 32.0, 34.0, 36.0]:
                for warning in [38.0, 45.0, 50.0, 55.0, 60.0]:
                    if warning <= danger: continue
                    states = hybrid_predict(X_val, classifier, anomaly_model, warning, danger, confidence, logic)['state']
                    m = safety_metrics(states, y_val)
                    m['utility'] = 2*m['recall_hazard'] + m['f1_hazard'] + m['balanced_accuracy_hazard'] - 2*m['non_hazard_fpr']
                    m.update({'contamination': contamination, 'logic': logic, 'confidence': confidence, 'danger_cm': danger, 'warning_cm': warning})
                    rows.append(m)
tuning = pd.DataFrame(rows).sort_values(['utility', 'recall_hazard', 'f1_hazard'], ascending=False).reset_index(drop=True)
best = tuning.iloc[0].to_dict()
print(tuning.head(10).to_string(index=False))
print('Selected validation configuration:', best)

selected_model = anomaly_models[float(best['contamination'])]
test_hybrid = hybrid_predict(X_test, classifier, selected_model, float(best['warning_cm']), float(best['danger_cm']), float(best['confidence']), str(best['logic']))
test_metrics = safety_metrics(test_hybrid['state'], y_test)
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))
print(test_hybrid['state'].value_counts())


In [ ]:
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'artifacts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
artifact = {'classifier': classifier, 'anomaly_model': selected_model, 'feature_columns': FEATURE_COLUMNS, 'label_names': LABELS, 'hazard_labels': sorted(HAZARD_LABELS), 'thresholds_cm': {'warning': float(best['warning_cm']), 'danger': float(best['danger_cm'])}, 'ai_confidence_threshold': float(best['confidence']), 'hybrid_logic': str(best['logic']), 'isolation_contamination': float(best['contamination']), 'feature_version': FEATURE_VERSION, 'target_policy': 'HAZARD=APPROACHING|DANGER|RETREATING; EMPTY=NO_OBJECT'}
joblib.dump(artifact, OUTPUT_DIR / 'hybrid_model_robust.joblib')
payload = {'validation_selected_config': best, 'test_metrics': test_metrics, 'test_class_support': y_test.value_counts().reindex(LABELS, fill_value=0).to_dict(), 'dropped_missing_sensor_rows': int(data.attrs['dropped_missing_sensor'])}
(OUTPUT_DIR / 'hybrid_metrics_robust.json').write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved:', OUTPUT_DIR / 'hybrid_model_robust.joblib')
print('Saved:', OUTPUT_DIR / 'hybrid_metrics_robust.json')


## Semantics khi triển khai

EMPTY là NO_OBJECT/non-hazard. Dòng thiếu left_cm hoặc right_cm đã bị loại trước feature engineering; không impute. Nếu dữ liệu online thiếu sensor, gateway nên trả về trạng thái chất lượng riêng và không đưa mẫu đó vào classifier như một mẫu bình thường.
